In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pm4py.objects.log.importer.xes import importer as xes_importer
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [19]:
file_path = "/Users/zeynepcetin/bppso-groupwork-1/data/BPI Challenge 2017.xes.gz"
print(f"Loading event log from: {file_path}")
log = xes_importer.apply(file_path)

print(f"Log loaded Total traces (cases): {len(log)}")

Loading event log from: /Users/zeynepcetin/bppso-groupwork-1/data/BPI Challenge 2017.xes.gz


parsing log, completed traces :: 100%|██████████| 31509/31509 [00:34<00:00, 914.36it/s] 

Log loaded Total traces (cases): 31509


In [20]:
case_data = []

for trace in log:
    attrs = trace.attributes
    
    # Extract specific fields, handling potential missing values
    case_data.append({
        "CaseID": attrs.get("concept:name", "Unknown"),
        "RequestedAmount": float(attrs.get("RequestedAmount", 0)),
        "CreditScore": float(attrs.get("CreditScore", 0)), # 0 usually means unknown/not calculated
        "LoanGoal": attrs.get("LoanGoal", "Unknown"),
        "ApplicationType": attrs.get("ApplicationType", "Unknown")
    })

df = pd.DataFrame(case_data)
print("Data extraction complete.")
display(df.head())

Data extraction complete.


,CaseID,RequestedAmount,CreditScore,LoanGoal,ApplicationType
0,Application_652823628,20000.0,0.0,Existing loan takeover,New credit
1,Application_1691306052,10000.0,0.0,Home improvement,New credit
2,Application_428409768,15000.0,0.0,Home improvement,New credit
3,Application_1746793196,5000.0,0.0,Car,New credit
4,Application_828200680,35000.0,0.0,Home improvement,New credit


In [21]:
print("--- RequestedAmount ---")

# Basic Stat
stats_amount = df['RequestedAmount'].describe()
print(stats_amount)

p05 = df['RequestedAmount'].quantile(0.05)
p95 = df['RequestedAmount'].quantile(0.95)

print(f"\n5th Percentile:  {p05:,.2f}")
print(f"95th Percentile: {p95:,.2f}")
print(f"Median:          {df['RequestedAmount'].median():,.2f}")


--- RequestedAmount ---
count     31509.000000
mean      16233.743989
std       15422.246299
min           0.000000
25%        6000.000000
50%       12500.000000
75%       21000.000000
max      450000.000000
Name: RequestedAmount, dtype: float64

5th Percentile:  0.00
95th Percentile: 46,000.00
Median:          12,500.00


In [22]:
print("--- CreditScore ---")

# Filter out 0
valid_scores = df[df['CreditScore'] > 0]['CreditScore']

print(f"Total Cases: {len(df)}")
print(f"Cases with valid CreditScore (>0): {len(valid_scores)}")
print(f"Percentage of known scores: {len(valid_scores)/len(df)*100:.2f}%")

if len(valid_scores) > 0:
    print("\nStatistics for Valid Scores:")
    print(valid_scores.describe())
else:
    print("No valid credit scores found in the dataset.")

# --> Delete the CreditScore from the Advanced Analysis

--- CreditScore ---
Total Cases: 31509
Cases with valid CreditScore (>0): 0
Percentage of known scores: 0.00%
No valid credit scores found in the dataset.


In [23]:
print("--- ApplicationType ---")
app_counts = df['ApplicationType'].value_counts()
print(app_counts)

print("--- LoanGoal ---")
goal_counts = df['LoanGoal'].value_counts()
print(goal_counts)

--- ApplicationType ---
ApplicationType
New credit     28120
Limit raise     3389
Name: count, dtype: int64
--- LoanGoal ---
LoanGoal
Car                       9328
Home improvement          7669
Existing loan takeover    5601
Other, see explanation    2985
Unknown                   2365
Not speficied             1065
Remaining debt home        842
Extra spending limit       625
Caravan / Camper           369
Motorcycle                 275
Boat                       201
Tax payments               152
Business goal               30
Debt restructuring           2
Name: count, dtype: int64
